# **Weather API ETL Pipeline**

- Extract > Open-Meteo API

- Transform > Pandas + data cleaning + feature engineering

- Load > SQLite + CSV

- Analyze > Pandas

- Visualize > Plotly

## Install/import libraries

In [1]:
!pip -q install requests pandas numpy plotly

In [2]:
# Standard libraries
import requests
import json
import sqlite3
import os
from datetime import datetime

# Data processing
import pandas as pd
import numpy as np

# Visualization
import plotly.express as px
import plotly.graph_objects as go

# Display
from IPython.display import display, HTML

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Define project configuration

Here we define the cities we want to monitor.

In [3]:
CITIES = [
    "Manila",
    "Cebu City",
    "Davao City",
    "Pasig City"
]

FORECAST_DAYS = 7

print("Cities:", CITIES)
print("Forecast horizon:", FORECAST_DAYS, "days")

Cities: ['Manila', 'Cebu City', 'Davao City', 'Pasig City']
Forecast horizon: 7 days


## Extract: Geocoding API

Before getting the weather data, we need to keep track of the coordinates such as latitude and longitude.

The API call converts the city name into coordinates.

In [6]:
GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"

In [7]:
def get_coordinates(city):
    """
    Convert a city name into latitude and longitude.
    """

    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(
        GEOCODING_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    if "results" not in data or len(data["results"]) == 0:
        raise ValueError(f"Could not find location: {city}")

    result = data["results"][0]

    return {
        "city": city,
        "latitude": result["latitude"],
        "longitude": result["longitude"],
        "country": result.get("country"),
        "timezone": result.get("timezone")
    }


locations = []

for city in CITIES:
    location = get_coordinates(city)
    locations.append(location)

locations_df = pd.DataFrame(locations)

display(locations_df)

,city,latitude,longitude,country,timezone
0,Manila,14.60420,120.98220,Philippines,Asia/Manila
1,Cebu City,10.31672,123.89071,Philippines,Asia/Manila
2,Davao City,7.07306,125.61278,Philippines,Asia/Manila
3,Pasig City,14.58691,121.06140,Philippines,Asia/Manila


Here, we just follow the workflow:

GET request -> Geocoding API -> JSON response -> Phyton dictionary -> Convert to Pandas Dataframe

## Inspect the raw API response

Next thing we do is to do data transformation and look upon the raw response.

In [11]:
test_location = locations[0]

params = {
    "latitude": test_location["latitude"],
    "longitude": test_location["longitude"],
    "hourly": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "rain",
        "weather_code",
        "wind_speed_10m"
    ]),
    "forecast_days": FORECAST_DAYS,
    "timezone": "auto"
}

response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params=params,
    timeout=30
)

response.raise_for_status()

raw_weather = response.json()

print(json.dumps(raw_weather, indent=2)[:5000])

{
  "latitude": 14.586995,
  "longitude": 121.002785,
  "generationtime_ms": 0.12767314910888672,
  "utc_offset_seconds": 28800,
  "timezone": "Asia/Manila",
  "timezone_abbreviation": "GMT+8",
  "elevation": 9.0,
  "hourly_units": {
    "time": "iso8601",
    "temperature_2m": "\u00b0C",
    "relative_humidity_2m": "%",
    "precipitation": "mm",
    "rain": "mm",
    "weather_code": "wmo code",
    "wind_speed_10m": "km/h"
  },
  "hourly": {
    "time": [
      "2026-09-20T00:00",
      "2026-09-20T01:00",
      "2026-09-20T02:00",
      "2026-09-20T03:00",
      "2026-09-20T04:00",
      "2026-09-20T05:00",
      "2026-09-20T06:00",
      "2026-09-20T07:00",
      "2026-09-20T08:00",
      "2026-09-20T09:00",
      "2026-09-20T10:00",
      "2026-09-20T11:00",
      "2026-09-20T12:00",
      "2026-09-20T13:00",
      "2026-09-20T14:00",
      "2026-09-20T15:00",
      "2026-09-20T16:00",
      "2026-09-20T17:00",
      "2026-09-20T18:00",
      "2026-09-20T19:00",
      "2026-09-20T

## Build the Weather API extraction function

Now we turn our API call into a reusable function.

In [12]:
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"


def get_weather_data(city, latitude, longitude, forecast_days=7):

    params = {
        "latitude": latitude,
        "longitude": longitude,

        "hourly": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "apparent_temperature",
            "precipitation_probability",
            "precipitation",
            "rain",
            "weather_code",
            "cloud_cover",
            "wind_speed_10m"
        ]),

        "forecast_days": forecast_days,

        # Automatically use the location's timezone
        "timezone": "auto"
    }

    response = requests.get(
        WEATHER_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    return data


print("Weather extraction function created.")

Weather extraction function created.


## Extract weather data for all cities

In [15]:
raw_data = {}

for location in locations:

    city = location["city"]

    print(f"Extracting weather for {city}...")

    data = get_weather_data(
        city=city,
        latitude=location["latitude"],
        longitude=location["longitude"],
        forecast_days=FORECAST_DAYS
    )

    raw_data[city] = data

print("\nExtraction completed.")
print("Locations extracted:", list(raw_data.keys()))

Extracting weather for Manila...
Extracting weather for Cebu City...
Extracting weather for Davao City...
Extracting weather for Pasig City...

Extraction completed.
Locations extracted: ['Manila', 'Cebu City', 'Davao City', 'Pasig City']


## Transform the nested JSON into pandas Dataframe

In [16]:
def transform_weather_data(city, raw_json):

    hourly = raw_json["hourly"]

    df = pd.DataFrame(hourly)

    # Add metadata
    df["city"] = city

    df["latitude"] = raw_json["latitude"]
    df["longitude"] = raw_json["longitude"]

    df["timezone"] = raw_json["timezone"]

    # Convert timestamp
    df["time"] = pd.to_datetime(df["time"])

    # Reorder columns
    columns = [
        "city",
        "time",
        "latitude",
        "longitude",
        "timezone",
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature",
        "precipitation_probability",
        "precipitation",
        "rain",
        "weather_code",
        "cloud_cover",
        "wind_speed_10m"
    ]

    df = df[columns]

    return df


weather_dfs = []

for city, data in raw_data.items():

    df_city = transform_weather_data(
        city,
        data
    )

    weather_dfs.append(df_city)


weather_df = pd.concat(
    weather_dfs,
    ignore_index=True
)

print("Rows:", len(weather_df))
print("Columns:", len(weather_df.columns))

display(weather_df.head())

Rows: 672
Columns: 14


,city,time,latitude,longitude,timezone,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation_probability,precipitation,rain,weather_code,cloud_cover,wind_speed_10m
0,Manila,2026-09-20 00:00:00,14.586995,121.002785,Asia/Manila,25.5,96,31.3,65,0.1,0.1,51,39,6.0
1,Manila,2026-09-20 01:00:00,14.586995,121.002785,Asia/Manila,25.3,96,30.8,51,0.0,0.0,2,63,6.9
2,Manila,2026-09-20 02:00:00,14.586995,121.002785,Asia/Manila,25.6,98,31.5,39,0.0,0.0,2,55,5.8
3,Manila,2026-09-20 03:00:00,14.586995,121.002785,Asia/Manila,25.3,97,31.1,30,0.0,0.0,3,81,5.4
4,Manila,2026-09-20 04:00:00,14.586995,121.002785,Asia/Manila,25.2,96,30.9,24,0.0,0.0,3,89,4.6


It is essential to know that database and analytics project usually benefit from consistent column names.

In [17]:
weather_df = weather_df.rename(columns={

    "temperature_2m": "temperature_c",

    "relative_humidity_2m": "humidity_pct",

    "apparent_temperature": "feels_like_c",

    "precipitation_probability":
        "precip_probability_pct",

    "precipitation":
        "precipitation_mm",

    "wind_speed_10m":
        "wind_speed_kmh"

})

display(weather_df.head())

,city,time,latitude,longitude,timezone,temperature_c,humidity_pct,feels_like_c,precip_probability_pct,precipitation_mm,rain,weather_code,cloud_cover,wind_speed_kmh
0,Manila,2026-09-20 00:00:00,14.586995,121.002785,Asia/Manila,25.5,96,31.3,65,0.1,0.1,51,39,6.0
1,Manila,2026-09-20 01:00:00,14.586995,121.002785,Asia/Manila,25.3,96,30.8,51,0.0,0.0,2,63,6.9
2,Manila,2026-09-20 02:00:00,14.586995,121.002785,Asia/Manila,25.6,98,31.5,39,0.0,0.0,2,55,5.8
3,Manila,2026-09-20 03:00:00,14.586995,121.002785,Asia/Manila,25.3,97,31.1,30,0.0,0.0,3,81,5.4
4,Manila,2026-09-20 04:00:00,14.586995,121.002785,Asia/Manila,25.2,96,30.9,24,0.0,0.0,3,89,4.6


## Feature Engineering

We will create the following features:

- date
- hour
- day of week
- heat index difference
- rain indicator
- high-temperature indicator

In [18]:
weather_df["date"] = weather_df["time"].dt.date

weather_df["hour"] = weather_df["time"].dt.hour

weather_df["day_of_week"] = weather_df["time"].dt.day_name()

weather_df["is_raining"] = (
    weather_df["precipitation_mm"] > 0
)

weather_df["temperature_difference"] = (
    weather_df["temperature_c"]
    - weather_df["feels_like_c"]
)

weather_df["is_hot"] = (
    weather_df["temperature_c"] >= 35
)

display(weather_df.head())

,city,time,latitude,longitude,timezone,temperature_c,humidity_pct,feels_like_c,precip_probability_pct,precipitation_mm,rain,weather_code,cloud_cover,wind_speed_kmh,date,hour,day_of_week,is_raining,temperature_difference,is_hot
0,Manila,2026-09-20 00:00:00,14.586995,121.002785,Asia/Manila,25.5,96,31.3,65,0.1,0.1,51,39,6.0,2026-09-20,0,Sunday,True,-5.8,False
1,Manila,2026-09-20 01:00:00,14.586995,121.002785,Asia/Manila,25.3,96,30.8,51,0.0,0.0,2,63,6.9,2026-09-20,1,Sunday,False,-5.5,False
2,Manila,2026-09-20 02:00:00,14.586995,121.002785,Asia/Manila,25.6,98,31.5,39,0.0,0.0,2,55,5.8,2026-09-20,2,Sunday,False,-5.9,False
3,Manila,2026-09-20 03:00:00,14.586995,121.002785,Asia/Manila,25.3,97,31.1,30,0.0,0.0,3,81,5.4,2026-09-20,3,Sunday,False,-5.8,False
4,Manila,2026-09-20 04:00:00,14.586995,121.002785,Asia/Manila,25.2,96,30.9,24,0.0,0.0,3,89,4.6,2026-09-20,4,Sunday,False,-5.7,False


## Weather condition mapping

In [19]:
weather_code_map = {

    0: "Clear sky",

    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",

    45: "Fog",
    48: "Depositing rime fog",

    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",

    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",

    71: "Slight snow",
    73: "Moderate snow",
    75: "Heavy snow",

    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",

    95: "Thunderstorm",
    96: "Thunderstorm with hail",
    99: "Thunderstorm with heavy hail"
}


weather_df["weather_description"] = (
    weather_df["weather_code"]
    .map(weather_code_map)
    .fillna("Unknown")
)

display(
    weather_df[
        [
            "city",
            "time",
            "temperature_c",
            "weather_code",
            "weather_description"
        ]
    ].head(10)
)

,city,time,temperature_c,weather_code,weather_description
0,Manila,2026-09-20 00:00:00,25.5,51,Light drizzle
1,Manila,2026-09-20 01:00:00,25.3,2,Partly cloudy
2,Manila,2026-09-20 02:00:00,25.6,2,Partly cloudy
3,Manila,2026-09-20 03:00:00,25.3,3,Overcast
4,Manila,2026-09-20 04:00:00,25.2,3,Overcast
5,Manila,2026-09-20 05:00:00,25.1,3,Overcast
6,Manila,2026-09-20 06:00:00,25.2,3,Overcast
7,Manila,2026-09-20 07:00:00,25.7,3,Overcast
8,Manila,2026-09-20 08:00:00,26.9,3,Overcast
9,Manila,2026-09-20 09:00:00,28.7,3,Overcast


## Data quality checks

We need to check the following:

- missing values
- duplicate rows
- invalid temperatures
- invalide humidity

In [23]:
# 1. Row count
print("\n1. Row count:")
print(len(weather_df))

# 2. Missing values
print("\n2. Missing values:")

missing = weather_df.isnull().sum()

display(
    missing[missing > 0]
)

# 3. Duplicate records
duplicates = weather_df.duplicated().sum()

print("\n3. Duplicate rows:", duplicates)

# 4. Temperature validation
invalid_temperature = weather_df[
    (weather_df["temperature_c"] < -80) |
    (weather_df["temperature_c"] > 70)
]

print(
    "\n4. Invalid temperature records:",
    len(invalid_temperature)
)

# 5. Humidity validation
invalid_humidity = weather_df[
    (weather_df["humidity_pct"] < 0) |
    (weather_df["humidity_pct"] > 100)
]

print(
    "5. Invalid humidity records:",
    len(invalid_humidity)
)

# 6. Precipitation validation
invalid_precip = weather_df[
    weather_df["precipitation_mm"] < 0
]

print(
    "6. Invalid precipitation records:",
    len(invalid_precip)
)


1. Row count:
672

2. Missing values:


,0



3. Duplicate rows: 0

4. Invalid temperature records: 0
5. Invalid humidity records: 0
6. Invalid precipitation records: 0


## Automated quality gates

Now we need to make the pipeline give us a warning if any serious problem occur along the way.

In [24]:
def run_quality_gate(df):

    assert len(df) > 0, \
        "ERROR: Dataset is empty."

    assert df["time"].notna().all(), \
        "ERROR: Missing timestamps."

    assert df["city"].notna().all(), \
        "ERROR: Missing city."

    assert df["temperature_c"].notna().all(), \
        "ERROR: Missing temperature."

    assert df["humidity_pct"].between(0, 100).all(), \
        "ERROR: Invalid humidity."

    assert (df["precipitation_mm"] >= 0).all(), \
        "ERROR: Negative precipitation."

    assert df.duplicated(
        subset=["city", "time"]
    ).sum() == 0, \
        "ERROR: Duplicate city/time records."

    print("✅ DATA QUALITY PASSED")


run_quality_gate(weather_df)

✅ DATA QUALITY PASSED


As you can see, we able to build a pipeline with validation controls.

## Load into SQLite

Since the project is for demonstration only of the ETL process, SQLite is perfect because it requires no server.

In [25]:
DATABASE_PATH = "weather_etl.db"

connection = sqlite3.connect(DATABASE_PATH)

weather_df.to_sql(
    "weather_hourly",
    connection,
    if_exists="replace",
    index=False
)

connection.commit()

print("Data loaded into:", DATABASE_PATH)

Data loaded into: weather_etl.db


## Query the database


In [26]:
query = """
SELECT
    city,
    time,
    temperature_c,
    humidity_pct,
    precipitation_mm,
    wind_speed_kmh
FROM weather_hourly
ORDER BY city, time
LIMIT 20
"""

db_df = pd.read_sql_query(
    query,
    connection
)

display(db_df)

,city,time,temperature_c,humidity_pct,precipitation_mm,wind_speed_kmh
0,Cebu City,2026-09-20 00:00:00,26.5,95,0.0,3.0
1,Cebu City,2026-09-20 01:00:00,26.3,94,0.0,1.9
2,Cebu City,2026-09-20 02:00:00,26.2,93,0.0,2.3
3,Cebu City,2026-09-20 03:00:00,25.8,94,0.0,3.2
4,Cebu City,2026-09-20 04:00:00,25.4,95,0.0,4.0
5,Cebu City,2026-09-20 05:00:00,25.1,95,0.0,3.0
6,Cebu City,2026-09-20 06:00:00,25.2,95,0.0,2.9
7,Cebu City,2026-09-20 07:00:00,27.0,92,0.0,3.5
8,Cebu City,2026-09-20 08:00:00,29.6,75,0.0,5.7
9,Cebu City,2026-09-20 09:00:00,30.1,72,0.2,9.3


## Create daily summary table

In [27]:
daily_df = (
    weather_df
    .groupby(["city", "date"])
    .agg(
        avg_temperature_c=(
            "temperature_c",
            "mean"
        ),

        max_temperature_c=(
            "temperature_c",
            "max"
        ),

        min_temperature_c=(
            "temperature_c",
            "min"
        ),

        avg_humidity_pct=(
            "humidity_pct",
            "mean"
        ),

        total_precipitation_mm=(
            "precipitation_mm",
            "sum"
        ),

        max_wind_speed_kmh=(
            "wind_speed_kmh",
            "max"
        ),

        avg_cloud_cover_pct=(
            "cloud_cover",
            "mean"
        )
    )
    .reset_index()
)

display(daily_df.head(10))

,city,date,avg_temperature_c,max_temperature_c,min_temperature_c,avg_humidity_pct,total_precipitation_mm,max_wind_speed_kmh,avg_cloud_cover_pct
0,Cebu City,2026-09-20,28.225000,31.9,25.1,83.916667,2.7,10.0,75.875000
1,Cebu City,2026-09-21,27.275000,30.6,25.0,89.125000,4.0,9.5,90.375000
2,Cebu City,2026-09-22,26.512500,29.7,24.5,91.875000,10.2,8.5,83.416667
3,Cebu City,2026-09-23,26.279167,28.9,24.8,93.541667,22.1,8.3,93.750000
4,Cebu City,2026-09-24,26.362500,29.4,24.7,92.750000,20.7,5.2,95.041667
5,Cebu City,2026-09-25,25.758333,28.0,24.1,94.583333,28.8,9.6,85.958333
6,Cebu City,2026-09-26,26.262500,28.7,23.2,90.541667,8.4,9.8,80.541667
7,Davao City,2026-09-20,27.504167,30.7,25.3,80.875000,0.4,16.6,87.500000
8,Davao City,2026-09-21,26.758333,30.6,24.7,82.958333,2.1,12.0,74.083333
9,Davao City,2026-09-22,26.454167,31.4,24.4,83.791667,7.1,12.8,78.541667


## Load daily table into database

In [28]:
daily_df.to_sql(
    "weather_daily",
    connection,
    if_exists="replace",
    index=False
)

connection.commit()

print("Daily summary loaded successfully.")

Daily summary loaded successfully.


## Export CSV files

In [29]:
weather_df.to_csv(
    "weather_hourly.csv",
    index=False
)

daily_df.to_csv(
    "weather_daily.csv",
    index=False
)

print("Files created:")
print("✓ weather_hourly.csv")
print("✓ weather_daily.csv")

Files created:
✓ weather_hourly.csv
✓ weather_daily.csv


## Basic Analytics

In [30]:
print("Average temperature by city:")

avg_temp = (
    weather_df
    .groupby("city")["temperature_c"]
    .mean()
    .sort_values(ascending=False)
)

display(avg_temp)

Average temperature by city:


,temperature_c
city,
Manila,26.919048
Pasig City,26.757738
Cebu City,26.667857
Davao City,26.632143


In [32]:
print("Total precipitation by city:")


rain_summary = (
    weather_df
    .groupby("city")["precipitation_mm"]
    .sum()
    .sort_values(ascending=False)
)

display(rain_summary)

Total precipitation by city:


,precipitation_mm
city,
Cebu City,96.9
Pasig City,77.1
Manila,39.7
Davao City,17.8


## Temperature Chart

In [33]:
fig = px.line(
    weather_df,
    x="time",
    y="temperature_c",
    color="city",
    title="Hourly Temperature Forecast",
    labels={
        "time": "Time",
        "temperature_c": "Temperature (°C)",
        "city": "City"
    }
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

## Humidity Chart

In [34]:
fig = px.line(
    weather_df,
    x="time",
    y="humidity_pct",
    color="city",
    title="Hourly Relative Humidity",
    labels={
        "time": "Time",
        "humidity_pct": "Humidity (%)",
        "city": "City"
    }
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

## Precipitation Chart

In [35]:
fig = px.line(
    weather_df,
    x="time",
    y="humidity_pct",
    color="city",
    title="Hourly Relative Humidity",
    labels={
        "time": "Time",
        "humidity_pct": "Humidity (%)",
        "city": "City"
    }
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

## Average temperature comparison

In [36]:
city_temperature = (
    daily_df
    .groupby("city")["avg_temperature_c"]
    .mean()
    .reset_index()
)

fig = px.bar(
    city_temperature,
    x="city",
    y="avg_temperature_c",
    title="Average Forecast Temperature by City",
    labels={
        "city": "City",
        "avg_temperature_c":
            "Average Temperature (°C)"
    }
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

## Dashboard KPI calculation

In [37]:
overall_avg_temp = weather_df[
    "temperature_c"
].mean()

overall_max_temp = weather_df[
    "temperature_c"
].max()

overall_rain = weather_df[
    "precipitation_mm"
].sum()

overall_avg_humidity = weather_df[
    "humidity_pct"
].mean()

rain_hours = weather_df[
    "is_raining"
].sum()

print("Average temperature:", round(overall_avg_temp, 2), "°C")
print("Maximum temperature:", round(overall_max_temp, 2), "°C")
print("Total precipitation:", round(overall_rain, 2), "mm")
print("Average humidity:", round(overall_avg_humidity, 2), "%")
print("Rain hours:", rain_hours)

Average temperature: 26.74 °C
Maximum temperature: 32.1 °C
Total precipitation: 231.5 mm
Average humidity: 87.06 %
Rain hours: 278


In [38]:
avg_temp = weather_df["temperature_c"].mean()
max_temp = weather_df["temperature_c"].max()
total_rain = weather_df["precipitation_mm"].sum()
avg_humidity = weather_df["humidity_pct"].mean()

# ------------------------------------------------------------
# KPI CARDS
# ------------------------------------------------------------

kpi_html = f"""
<div style="
    display:flex;
    gap:20px;
    margin-bottom:25px;
">

<div style="
    flex:1;
    padding:20px;
    border-radius:10px;
    background:#f2f2f2;
    text-align:center;
">
<h3>Average Temperature</h3>
<h1>{avg_temp:.1f} °C</h1>
</div>

<div style="
    flex:1;
    padding:20px;
    border-radius:10px;
    background:#f2f2f2;
    text-align:center;
">
<h3>Maximum Temperature</h3>
<h1>{max_temp:.1f} °C</h1>
</div>

<div style="
    flex:1;
    padding:20px;
    border-radius:10px;
    background:#f2f2f2;
    text-align:center;
">
<h3>Total Rain</h3>
<h1>{total_rain:.1f} mm</h1>
</div>

<div style="
    flex:1;
    padding:20px;
    border-radius:10px;
    background:#f2f2f2;
    text-align:center;
">
<h3>Average Humidity</h3>
<h1>{avg_humidity:.1f}%</h1>
</div>

</div>
"""

display(HTML(kpi_html))

In [39]:
fig_temp = px.line(
    weather_df,
    x="time",
    y="temperature_c",
    color="city",
    title="🌡️ 7-Day Temperature Forecast"
)

fig_temp.update_layout(
    template="plotly_white",
    height=500,
    hovermode="x unified"
)

fig_temp.show()

In [40]:
fig_rain = px.bar(
    daily_df,
    x="date",
    y="total_precipitation_mm",
    color="city",
    barmode="group",
    title="🌧️ Daily Precipitation"
)

fig_rain.update_layout(
    template="plotly_white",
    height=500
)

fig_rain.show()

In [41]:
fig_humidity = px.line(
    weather_df,
    x="time",
    y="humidity_pct",
    color="city",
    title="💧 Relative Humidity"
)

fig_humidity.update_layout(
    template="plotly_white",
    height=500,
    hovermode="x unified"
)

fig_humidity.show()

In [43]:
condition_summary = (
    weather_df
    .groupby(
        ["city", "weather_description"]
    )
    .size()
    .reset_index(name="hours")
)

display(
    condition_summary.sort_values(
        ["city", "hours"],
        ascending=[True, False]
    ).sort_values("city")
)

,city,weather_description,hours
4,Cebu City,Overcast,56
1,Cebu City,Light drizzle,45
5,Cebu City,Partly cloudy,18
7,Cebu City,Thunderstorm,17
3,Cebu City,Moderate drizzle,13
2,Cebu City,Mainly clear,6
0,Cebu City,Dense drizzle,5
8,Cebu City,Thunderstorm with hail,5
6,Cebu City,Slight rain showers,3
14,Davao City,Moderate rain showers,1


## Most common weather condition

In [44]:
most_common_conditions = (
    weather_df
    .groupby(
        ["city", "weather_description"]
    )
    .size()
    .reset_index(name="hours")
    .sort_values(
        ["city", "hours"],
        ascending=[True, False]
    )
)

most_common_conditions = (
    most_common_conditions
    .groupby("city")
    .head(1)
)

display(most_common_conditions)

,city,weather_description,hours
4,Cebu City,Overcast,56
15,Davao City,Overcast,78
20,Manila,Mainly clear,46
28,Pasig City,Light drizzle,37


## Summary

In [45]:
summary = (
    weather_df
    .groupby("city")
    .agg(
        avg_temperature_c=(
            "temperature_c",
            "mean"
        ),

        max_temperature_c=(
            "temperature_c",
            "max"
        ),

        avg_humidity_pct=(
            "humidity_pct",
            "mean"
        ),

        total_rain_mm=(
            "precipitation_mm",
            "sum"
        ),

        max_wind_kmh=(
            "wind_speed_kmh",
            "max"
        )
    )
    .round(2)
    .reset_index()
)

display(summary)

,city,avg_temperature_c,max_temperature_c,avg_humidity_pct,total_rain_mm,max_wind_kmh
0,Cebu City,26.67,31.9,90.90,96.9,10.0
1,Davao City,26.63,31.4,84.52,17.8,16.6
2,Manila,26.92,31.3,85.82,39.7,13.5
3,Pasig City,26.76,32.1,87.00,77.1,12.4


## Close database

In [46]:
connection.close()

print("SQLite connection closed.")
print("ETL pipeline completed successfully! ✅")

SQLite connection closed.
ETL pipeline completed successfully! ✅
